# **Exercises**


##### 1. Embed 5 sentences of your own (2 similar, 3 different). Print the similarity matrix and check whether the scores match your intuition.

In [1]:
import warnings
import numpy as np
import tiktoken
from pypdf import PdfReader
warnings.filterwarnings("ignore")
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
sentences = [
    "The weather is lovely and sunny today.",
    "It is a bright and beautiful sunny day outside.",
    "Python is a popular programming language.",
    "I love eating hot pizza with extra cheese.",
    "Quantum computing will change the future of technology."
]

# Load the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for the sentences
embeddings = model.encode(sentences)

# Compute the similarity matrix
simarity_matrix = cosine_similarity(embeddings)

print("Cosine Similarity Matrix:")
print(simarity_matrix)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2537.33it/s]


Cosine Similarity Matrix:
[[ 0.9999997   0.85846585  0.070928    0.11372606  0.0034641 ]
 [ 0.85846585  1.0000004   0.05649678  0.12553023 -0.01596536]
 [ 0.070928    0.05649678  1.          0.09412867  0.21033126]
 [ 0.11372606  0.12553023  0.09412867  1.         -0.01347312]
 [ 0.0034641  -0.01596536  0.21033126 -0.01347312  0.9999997 ]]


##### 2. Take a paragraph from any PDF and count its tokens with `tiktoken`. Compare with `len(text.split())`.

In [4]:
# read the PDF file
reader = PdfReader("C:\\Users\\arya9\\Documents\\Final_FREQUENTLY_ASKED_QUESTIONS_-PATENT.pdf")
# Generate embeddings for the paragraphs

# Extract text from the first page 
page = reader.pages[0]
text = page.extract_text()

# Take a single paragraph from the text
paragraph = text.split("\n")[0]

# Count tokens using tiktoken
encoding = tiktoken.encoding_for_model("gpt-4o")
paragraph_tokens = len(encoding.encode(paragraph))

# Count words using Python's split()
word_count = len(paragraph.split())

print(f"Paragraph: {paragraph}\n")
print(f"Token Count (tiktoken): {paragraph_tokens}")
print(f"Word Count (len.split()): {word_count}")

Paragraph: Office of CGPDTM, INDIA | www.ipindia.gov.in Page 1  

Token Count (tiktoken): 18
Word Count (len.split()): 8


##### 3. Pick a model with a known context window and calculate how many 500-token chunks fit after reserving space for the prompt and answer.

**Model Chosen:** GPT-4o

**Total Context Window:** 128,000 tokens

**Reserved Space:** 3,000 tokens (for prompt and answer combined)

**Available Space for Chunks:** 125,000 tokens (128,000 - 3,000)

**Chunk Size:** 500 tokens per chunk

**Total Chunks that Fit:** 250 chunks (125,000 / 500)

##### 4. Run an example with your own 5 to 10 sentences. Try questions that **reuse no words** from the documents and see if retrieval still works.

In [5]:
#  Define 5-10 sentences representing your "document" knowledge base
documents = [
    "The central bank decided to hike interest rates by 50 basis points to combat inflation.",
    "Feline pets require high-protein diets and regular veterinary checkups for optimal health.",
    "Electric vehicles rely on lithium-ion battery packs to store electrical energy.",
    "Astronomers discovered a rocky exoplanet orbiting within the habitable zone of a distant star.",
    "Marathon runners need to carb-load and maintain proper hydration before race day."
]

# Load a pre-trained sentence embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for the documents
doc_embeddings = model.encode(documents)

# query that shares NO exact words with the documents
query = ["How do cats stay healthy?"]

# Generate embeddings for the query
query_embeddings = model.encode(query)

# Compute cosine similarity between the query and each document
similarities = cosine_similarity(query_embeddings, doc_embeddings)

# Flatten the similarity array (since cosine_similarity returns a 2D array like [[score1, score2, ...]])
flat_similarities = similarities[0]

# Find the index of the document with the highest score
best_match_index = np.argmax(flat_similarities)

# Retrieve the best score and the corresponding document
best_score = flat_similarities[best_match_index]
best_answer = documents[best_match_index]

# the final retrieved answer
print(f"Query: {query[0]}\n")
print(f"Retrieved Best Match (Score: {best_score:.4f}):")
print(f"-> {best_answer}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 666.55it/s]


Query: How do cats stay healthy?

Retrieved Best Match (Score: 0.5235):
-> Feline pets require high-protein diets and regular veterinary checkups for optimal health.


##### 5. Take the two functions in section 5 and run them on a small PDF of your own. Print the retrieved chunks for 3 different questions.

In [8]:
# 1: Load and Clean PDF Text

def load_and_clean_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    clean_lines = []
    
    for page in reader.pages:
        text = page.extract_text()
        if not text:
            continue
        for line in text.split("\n"):
            line_stripped = line.strip()
            # Skip empty lines, headers, or footers
            if not line_stripped:
                continue
            if "www." in line_stripped.lower() or "page" in line_stripped.lower():
                continue
            clean_lines.append(line_stripped)
            
    return " ".join(clean_lines)

In [7]:
# Token-Based Chunking Function
def chunk_text_by_tokens(text, max_tokens=500, overlap=100):
    encoding = tiktoken.encoding_for_model("gpt-4o")
    tokens = encoding.encode(text)
    
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = encoding.decode(chunk_tokens)
        chunks.append(chunk_text)
        start += max_tokens - overlap  # Move forward with overlap
        
    return chunks

In [ ]:
# STEP-BY-STEP EXECUTION

#  Load your PDF (Replace with your actual PDF file name)
pdf_filename = "C:\\Users\\arya9\\Documents\\Final_FREQUENTLY_ASKED_QUESTIONS_-PATENT.pdf" 
# (For testing right now, you can also replace pdf_text with a long string if your PDF isn't ready)
try:
    raw_text = load_and_clean_pdf(pdf_filename)
except Exception:
    # Fallback dummy text if the file isn't found yet
    raw_text = (
        "Artificial intelligence is transforming industries worldwide. "
        "Machine learning models require large datasets to train effectively. "
        "Central banks adjust interest rates to control inflation and stabilize economies. "
        "Electric vehicles rely heavily on advanced lithium-ion battery technology. "
        "Space exploration has revealed rocky exoplanets in distant stellar habitable zones."
    )

# 2. Chunk the text
document_chunks = chunk_text_by_tokens(raw_text, max_tokens=50, overlap=10)
print(f"Total Chunks Created: {len(document_chunks)}\n")

# 3. Load embedding model and embed chunks
model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = model.encode(document_chunks)

# 4. Define 3 different questions to test retrieval
queries = [
    "How do financial institutions manage inflation?",
    "What powers modern electric cars?",
    "Tell me about machine learning training requirements."
]

# 5. Run retrieval for each question
for i, q in enumerate(queries, 1):
    query_embedding = model.encode([q])
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]
    
    best_match_idx = np.argmax(similarities)
    best_score = similarities[best_match_idx]
    retrieved_chunk = document_chunks[best_match_idx]
    
    print(f"--- Query {i}: {q} ---")
    print(f"Score: {best_score:.4f}")
    print(f"Retrieved Chunk: {retrieved_chunk}\n")